In [5]:

!pip  install seaborn matplotlib pandas numpy
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
# Import du jeu de donnée asssemblé mais non nettoyé
df = pd.read_csv("../data/raw/accidents_2019_2023.csv")


C:\Users\nvann\AppData\Local\Temp\ipykernel_34924\1026929783.py:7: DtypeWarning: Columns (42,45,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/accidents_2019_2023.csv")


In [6]:
#Étape 1 — Nettoyer les données
df[['secu1', 'secu2', 'secu3']] = df[['secu1', 'secu2', 'secu3']].replace(-1, np.nan)
# Étape 2 — Rassembler les équipements observés
#  On concatène les colonnes secu en une seule série d’observations
secu_all = pd.concat([
    df[['catu', 'catv', 'secu1']].rename(columns={'secu1': 'secu'}),
    df[['catu', 'catv', 'secu2']].rename(columns={'secu2': 'secu'})
], axis=0)
#Étape 3 — Calcul de la distribution conditionnelle
# Profil = combinaison de catu et catv (ex : "1_7")
secu_all['profil'] = secu_all['catu'].astype(str) + '_' + secu_all['catv'].astype(str)

# Calcul des distributions conditionnelles
proba_secu_par_profil = (
    secu_all.groupby('profil')['secu']
    .value_counts(normalize=True)
    .unstack(fill_value=0)
)

def imputer_secu(df, col, proba_df, profil_cols):
    def impute(row):
        if pd.notna(row[col]):
            return row[col]
        # Génère une clé profil (ex : "1_7")
        profil_key = "_".join([str(row[c]) for c in profil_cols])
        if profil_key not in proba_df.index:
            return np.nan  # ou fallback global
        p = proba_df.loc[profil_key]
        return np.random.choice(p.index, p=p.values)
    return df.apply(impute, axis=1)

# Générer le profil dans le dataframe principal
df['profil'] = df['catu'].astype(str) + '_' + df['catv'].astype(str)

# Imputation des 3 colonnes
df['secu1_corr'] = imputer_secu(df, 'secu1', proba_secu_par_profil, ['catu', 'catv'])
df['secu2_corr'] = imputer_secu(df, 'secu2', proba_secu_par_profil, ['catu', 'catv'])
df['secu3_corr'] = imputer_secu(df, 'secu3', proba_secu_par_profil, ['catu', 'catv'])
#On regroupe
df['equipements'] = df[['secu1_corr', 'secu2_corr', 'secu3_corr']].values.tolist()

In [7]:
df[['catu', 'catv', 'secu1', 'secu2', 'secu3', 'secu1_corr', 'secu2_corr', 'secu3_corr', 'equipements']].to_csv("../data/processed/accidents_2019_2023_secu.csv", index=False)

PermissionError: [Errno 13] Permission denied: '../data/processed/accidents_2019_2023_secu.csv'

In [ ]:
import numpy as np
import pandas as pd

# Étape 1 — Nettoyer les données (remplacer -1 par NaN)
df[['secu1', 'secu2']] = df[['secu1', 'secu2']].replace(-1, np.nan)

# Étape 2 — Fonction pour calculer la distribution conditionnelle par colonne
def get_proba_by_col(df, colname, group_cols=['catu', 'catv']):
    temp = df[df[colname].notna()].copy()
    temp['profil'] = temp[group_cols].astype(str).agg('_'.join, axis=1)
    return (
        temp.groupby('profil')[colname]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
    )

# Étape 3 — Calcul des distributions conditionnelles séparées
proba_secu1 = get_proba_by_col(df, 'secu1')
proba_secu2 = get_proba_by_col(df, 'secu2')


# Étape 4 — Fonction d’imputation basée sur les distributions
def imputer_secu(df, col, proba_df, profil_cols):
    def impute(row):
        if pd.notna(row[col]):
            return row[col]
        profil_key = "_".join([str(row[c]) for c in profil_cols])
        if profil_key not in proba_df.index:
            return np.nan  # fallback possible ici
        p = proba_df.loc[profil_key]
        return np.random.choice(p.index, p=p.values)
    return df.apply(impute, axis=1)

# Étape 5 — Imputation indépendante des 3 colonnes
df['secu1_corr'] = imputer_secu(df, 'secu1', proba_secu1, ['catu', 'catv'])
df['secu2_corr'] = imputer_secu(df, 'secu2', proba_secu2, ['catu', 'catv'])


# Étape 6 — Regrouper les équipements corrigés dans une seule liste
def regrouper_equipements(row):
    return [x for x in [row['secu1_corr'], row['secu2_corr']] if pd.notna(x)]

df['equipements'] = df.apply(regrouper_equipements, axis=1)
d

NameError: name 'df' is not defined

In [9]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
equip_ohe = pd.DataFrame(mlb.fit_transform(df['equipements']), 
                         columns=[f'eq_{int(c)}' for c in mlb.classes_],
                         index=df.index)

# Fusion avec ton DataFrame principal
df = pd.concat([df, equip_ohe], axis=1)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 619025 entries, 0 to 619024
Data columns (total 69 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Num_Acc      619025 non-null  int64  
 1   id_vehicule  619025 non-null  object 
 2   num_veh      619025 non-null  object 
 3   senc         619025 non-null  int64  
 4   catv         619025 non-null  int64  
 5   obs          619025 non-null  int64  
 6   obsm         619025 non-null  int64  
 7   choc         619025 non-null  int64  
 8   manv         619025 non-null  int64  
 9   motor        619025 non-null  int64  
 10  occutc       7751 non-null    float64
 11  place        619025 non-null  int64  
 12  catu         619025 non-null  int64  
 13  grav         619025 non-null  int64  
 14  sexe         619025 non-null  int64  
 15  an_nais      610486 non-null  float64
 16  trajet       619025 non-null  int64  
 17  secu1        611144 non-null  float64
 18  secu2        370822 non-

In [12]:

df[['catu', 'catv', 'secu1', 'secu2', 'secu3', 'secu1_corr', 'secu2_corr', 'secu3_corr', 'equipements', 'eq_1','eq_2','eq_3','eq_4','eq_5','eq_6','eq_7','eq_8','eq_9']].to_csv("../data/processed/accidents_2019_2023_secu.csv", index=False)

In [ ]:
f.drop(columns=['secu1', 'secu2','secu3','secu1_corr','secu1_corr','secu1_corr', 'equipements'], inplace=True)